# 01 — Evaluate a black-box retriever

**Companion to Lessons 1 & 2.**

> *Imagine you're building a recruiting application. You have a
> pool of résumés in a database and a job opening to fill. A
> hiring manager needs a shortlist by the end of the day. You run
> your retrieval system and get back a ranked list of candidates.*
> *But how do you know if it's any good?*

That's the question this notebook answers, end-to-end, on real
data. The "retrieval system" here is **lexical (BM25) search** —
the same family of keyword-matching algorithms that has powered
full-text search engines for decades. We treat it as a black box:
text goes in, a ranked list of candidate documents comes out. Our
job is to score that ranked list against ground-truth relevance
judgements.

## What you'll do

1. Load a BEIR dataset (the same one you ingested in notebook 00)
   and look at one of its queries and its qrels.
2. Run BM25 retrieval for that query.
3. Compute **Precision@k**, **Recall@k**, **NDCG@k**, and **MRR**
   by hand on the result — once you've felt how the formulas
   work, you'll use the library functions for the rest.
4. Loop over many queries and aggregate the metrics into a single
   score for the system.

In [ ]:
import os, sys
# Notebooks live in agent-notebooks/. Add the repo root to the path so
# we can import the lab's library modules.
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

In [ ]:
# Make sure DATASET matches what you ingested in notebook 00.
DATASET = "scifact"

import pymongo
from lib import MONGODB_URI, DB_NAME, collection_name, load_beir_dataset

client = pymongo.MongoClient(MONGODB_URI)
coll   = client[DB_NAME][collection_name(DATASET)]
corpus, queries, qrels, info = load_beir_dataset(DATASET)

print(f"Dataset      : {DATASET}  ({info['description']})")
print(f"Corpus       : {len(corpus):,} documents")
print(f"Test queries : {len(queries):,}")
print(f"Collection   : {coll.estimated_document_count():,} chunks ingested")

## What is an evaluation dataset?

Every BEIR dataset is a triple:

- **`corpus`** — the haystack: a `dict` of `{doc_id: {"title": ..., "text": ...}}`.
- **`queries`** — what users would ask: a `dict` of `{query_id: query_text}`.
- **`qrels`** ("query relevance" judgements) — the answer key: a
  `dict` of `{query_id: {doc_id: relevance_score}}`. A non-zero
  score means a human judged that document relevant to that query.

The qrels are sometimes called the **judgement list**, **golden
dataset**, or **ground truth**. They're what makes evaluation
possible: without a known answer, we can't measure error.

Let's pull one query and look at it.

In [ ]:
from retrieve import text_only

# Pick the first query whose BM25 top-10 actually contains at least
# one of its relevant docs — otherwise every metric below would
# be 0. (Queries that share no keywords with their relevant docs
# are great examples of where vector beats lexical, but they make a
# confusing first metric walkthrough.)
def has_bm25_hit(qid):
    q_qrels = qrels.get(qid, {})
    rel_set = {did for did, s in q_qrels.items() if s > 0}
    if not rel_set:
        return False
    ranked = text_only(coll, queries[qid], top_k=10)
    return any(r['doc_id'] in rel_set for r in ranked)

sample_qid    = next(qid for qid in queries if has_bm25_hit(qid))
sample_query  = queries[sample_qid]
sample_qrels  = qrels.get(sample_qid, {})
relevant_docs = {did: s for did, s in sample_qrels.items() if s > 0}

print(f"Query ID     : {sample_qid}")
print(f"Query text   : {sample_query!r}")
print(f"Relevant docs (per the qrels): {len(relevant_docs)}")
for did, score in list(relevant_docs.items())[:5]:
    title = corpus.get(did, {}).get('title', '<not in corpus>')
    print(f"  doc {did:<12}  relevance={score}  title={title!r}")

### Graded vs binary relevance

Notice the qrels are numbers, not booleans. BEIR datasets use
**graded relevance**:

- `0` — not relevant
- `1` — relevant
- `2` — highly relevant (in datasets that support it)

Binary relevance is graded relevance forced to `{0, 1}`. Most
metrics work with either; **NDCG** is the one that benefits the
most from the extra resolution (we'll see why below).

## Run the black-box retriever

We'll call `retrieve.text_only(...)` — pure BM25 search via
MongoDB Atlas's `$search`. Input: the query string. Output: a
ranked list of documents, best-first, with a score.

In [ ]:
from retrieve import text_only

TOP_K = 10
ranked = text_only(coll, sample_query, top_k=TOP_K)

print(f"Top {TOP_K} BM25 results for query {sample_qid!r}:")
print(f"  {sample_query!r}")
print()
for rank, row in enumerate(ranked, 1):
    grade = sample_qrels.get(row['doc_id'], 0)
    tag = "★" if grade > 0 else " "
    print(f"  {rank:>2}. {tag}  doc {row['doc_id']:<12}  "
          f"score={row['score']:.3f}  grade={grade}  "
          f"{row['title'][:60]!r}")

Each row marked ★ is one the qrels say is relevant; unmarked rows
are either confirmed irrelevant (grade 0) or not judged at all.
Take a moment to read the unmarked results — would *you* call any
of them relevant? That gut check is what motivates **graded** vs
**binary** relevance and the careful curation that goes into
qrels.

## Metric 1 — Precision@k

> *Of the documents the system returned, what fraction are relevant?*

Precision answers the **false positive** question: "how much
noise is in my shortlist?" If we returned 5 candidates and 4 of
them are relevant, Precision@5 = 0.8.

$$\text{Precision@k} = \frac{|\text{relevant} \cap \text{top-}k|}{k}$$

In [ ]:
# By hand for k=5
top5 = [row['doc_id'] for row in ranked[:5]]
relevant_ids = {did for did, s in sample_qrels.items() if s > 0}

hits_in_top5 = sum(1 for did in top5 if did in relevant_ids)
precision_at_5 = hits_in_top5 / 5

print(f"Top-5 doc IDs       : {top5}")
print(f"Relevant in top-5   : {hits_in_top5}")
print(f"Precision@5         : {precision_at_5:.3f}")

# Cross-check with the library
from lib_metrics import precision_at_k
print(f"Library Precision@5 : {precision_at_k(top5, sample_qrels, 5):.3f}")

## Metric 2 — Recall@k

> *Of all the relevant documents that exist, what fraction did we
> return?*

Recall answers the **false negative** question: "how many
relevant documents did I miss?" If 10 docs in the corpus are
relevant and 7 of them appear in the top-k, Recall@k = 0.7.

$$\text{Recall@k} = \frac{|\text{relevant} \cap \text{top-}k|}{|\text{relevant}|}$$

> **Vocabulary note.** "Recall" also shows up in *vector index*
> benchmarks where it means the fraction of true nearest
> neighbours an approximate index returned. Same word, different
> reference point — that's an index-quality metric, this is a
> retrieval-quality metric.

In [ ]:
from lib_metrics import recall_at_k

top10 = [row['doc_id'] for row in ranked[:10]]
hits_in_top10 = len(set(top10) & relevant_ids)

print(f"# relevant docs in corpus : {len(relevant_ids)}")
print(f"# relevant in top-10      : {hits_in_top10}")
print(f"Recall@10  (by hand)      : {hits_in_top10 / max(1, len(relevant_ids)):.3f}")
print(f"Recall@10  (library)      : {recall_at_k(top10, sample_qrels, 10):.3f}")

### Precision/Recall trade-off

These two metrics pull in opposite directions:

- **Want higher precision?** Return fewer, higher-confidence
  results. You'll miss more — recall drops.
- **Want higher recall?** Cast a wider net. You'll pull in more
  noise — precision drops.

Which one matters more is an **application-level** decision. A
legal-discovery tool can't afford to miss a relevant precedent
(favour recall). A user-facing search box can't afford to bury
useful results under junk (favour precision). The same retriever
can be the right or wrong tool depending on the cost of each
error type.

## Metric 3 — NDCG@k

Precision and Recall both treat the top-k as a *set* — they
ignore the order within it. But users look at ranked lists
top-down; a relevant document at rank 1 is much more valuable
than the same document at rank 9. **NDCG** captures that.

NDCG ("Normalized Discounted Cumulative Gain") has three pieces:

1. **Gain** — the relevance grade of each retrieved document
   (so a graded judgement of 2 contributes more than a 1).
2. **Discount** — divide the gain by `log₂(rank + 1)`, so
   position 1 gets weight `1 / log₂(2) = 1.0`, position 5 gets
   `1 / log₂(6) ≈ 0.39`, position 10 gets `≈ 0.29`. Lower
   positions count less.
3. **Normalization** — divide by the score of an *ideal*
   ranking (relevant docs sorted from highest grade to lowest).
   That bounds NDCG to `[0, 1]`. A perfect ranking scores 1.0.

$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{grade}(d_i)}{\log_2(i+1)}
\quad\quad \text{NDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}$$

In [ ]:
import math
from lib_metrics import ndcg_at_k, dcg_at_k

# Compute DCG@5 by hand
ranked_ids_5 = [row['doc_id'] for row in ranked[:5]]
print("DCG calculation (k=5):")
dcg = 0.0
for rank, did in enumerate(ranked_ids_5, 1):
    grade = sample_qrels.get(did, 0)
    disc  = math.log2(rank + 1)
    term  = grade / disc
    dcg  += term
    print(f"  rank {rank}: grade={grade}  discount=1/log2({rank+1})={1/disc:.3f}  term={term:.3f}")
print(f"  → DCG@5 = {dcg:.3f}")

# And IDCG@5 — the DCG of the best possible ranking
ideal_grades = sorted(sample_qrels.values(), reverse=True)[:5]
idcg = sum(g / math.log2(i + 1) for i, g in enumerate(ideal_grades, 1) if g > 0)
print(f"  → IDCG@5 = {idcg:.3f}  (best possible ordering of available grades)")
print(f"  → NDCG@5 = DCG/IDCG = {dcg/idcg if idcg else 0:.3f}")
print()
print(f"Library NDCG@5  : {ndcg_at_k(ranked_ids_5, sample_qrels, 5):.3f}")
print(f"Library NDCG@10 : {ndcg_at_k([r['doc_id'] for r in ranked[:10]], sample_qrels, 10):.3f}")

NDCG@k is **the** primary metric for embedding-model comparison;
you'll see it on every public leaderboard (MTEB, RTEB, BEIR
itself). It rewards systems that put the most relevant
documents at the top and tolerates burying weaker (but
still-relevant) ones lower.

## Metric 4 — MRR (Mean Reciprocal Rank)

> *On average, how high up does the first relevant document
> appear?*

Scan the ranked list from position 1 down until you hit a
relevant doc. If it's at rank `r`, the reciprocal rank is
`1/r`: 1.0 at position 1, 0.5 at position 2, 0.33 at position
3, etc. Average across all queries to get MRR.

MRR is most informative when there's **one** clearly best fit
per query, or when you're evaluating a **reranker** whose only
job is to bubble the best answer to the top.

In [ ]:
from lib_metrics import reciprocal_rank

ranked_ids = [row['doc_id'] for row in ranked]
first_rel_rank = next(
    (i for i, did in enumerate(ranked_ids, 1) if did in relevant_ids),
    None,
)
if first_rel_rank:
    print(f"First relevant doc at rank {first_rel_rank}  →  RR = 1/{first_rel_rank} = {1/first_rel_rank:.3f}")
else:
    print(f"No relevant doc retrieved at all  →  RR = 0")

print(f"Library Reciprocal Rank : {reciprocal_rank(ranked_ids, sample_qrels):.3f}")

## Putting it all together: evaluate over many queries

A retrieval system isn't judged on one query — it's judged on a
distribution of queries. We'll loop over the first `N` test
queries, compute the per-query metric dict for each, and
aggregate by averaging across queries (AP becomes MAP, others
stay as means).

In [ ]:
from lib import embed_queries
from lib_metrics import compute_query_metrics, aggregate_metrics, format_summary

N_QUERIES = 30   # bump up once you've confirmed it works
TOP_K     = 10

# Use a stable order so re-runs evaluate the same queries
query_ids = list(queries.keys())[:N_QUERIES]

per_query: list[dict] = []
for qid in query_ids:
    q_text = queries[qid]
    ranked = text_only(coll, q_text, top_k=TOP_K)
    ranked_ids = [row['doc_id'] for row in ranked]
    metrics = compute_query_metrics(ranked_ids, qrels.get(qid, {}))
    per_query.append(metrics)

agg = aggregate_metrics(per_query)
print(f"BM25 lexical retrieval — {DATASET}, {N_QUERIES} queries")
print(format_summary(agg))

## Reading the aggregate

Each metric on that single line tells you something different.
For a quick gut-check:

- **P@5 / P@10** — how clean the top of the list is. Higher =
  fewer false positives in what the user actually sees.
- **R@5 / R@10** — coverage. Higher = fewer false negatives in
  the top-k window. Watch this when missing a relevant doc is
  costly (legal, medical, compliance).
- **NDCG@5 / NDCG@10** — ranking quality. The number you'd
  quote when comparing two retrievers on a benchmark.
- **MRR** — how prominently the *first* relevant result sits.
- **MAP** — Mean Average Precision; a recall-and-rank summary
  across all positions where relevant docs land.

Save these numbers — you'll compare them to a different black
box in notebook 02.

## Next

Open **`02_swap_blackbox.ipynb`** to swap BM25 for vector and
hybrid retrieval over the same dataset, and see how the metric
deltas explain *when* and *why* one approach beats another.